# 02：搭建量子模型的训练循环

定义一个单量子比特模型，选择目标，查看梯度，然后亲自更新参数。
接着，用一个简短的 PyTorch 循环重复这个过程。
这里没有调用训练辅助脚本，算法的每个环节都能在 notebook 中直接看到。

[English edition](../notebooks/02_quantum_training.ipynb)

**学习目标：** 用标准 PyTorch 梯度与优化器循环训练量子门角度。

**先修要求：** 已完成 01，并了解 PyTorch 张量、梯度和优化器。全部计算在本地 CPU 上进行，不需要真机账号。

学习路线见 [workshop 导航](../README.md)。


## 1. 定义带有可调角度的电路

RY 门用角度 `theta` 旋转一个量子比特。下面的函数描述如何根据当前参数构造电路。
可选的 `inputs` 参数在这个小示例中不使用；在更大的模型中，它可以承载输入数据。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import flagquantum as fq
import torch


def build_circuit(parameters, inputs=None):
    circuit = fq.Circuit(1)
    circuit.ry(0, parameters["theta"][0])
    return circuit


## 2. 让角度成为可训练参数

`fq.Module` 管理名为 `theta` 的 PyTorch 可训练参数。形状 `(1,)` 表示一个数。
设置随机种子后，重新运行此单元会得到相同的初始值。

模型返回量子比特 0 的 Z 期望值。Z 测量将 `0` 记为 +1，将 `1` 记为 -1；期望值是它们按概率加权的平均。
对这个电路，答案是 `cos(theta)`。

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。 这里只画初始参数下的电路，不在训练循环中重复绘图。


In [ ]:
policy = fq.RuntimePolicy(
    execution_options=fq.ExecutionOptions(device="cpu", mode="statevector"),
    observable="z_sum",
    observable_qubits=(0,),
)
model = fq.Module(
    build_circuit,
    parameters={"theta": (1,)},
    init="uniform",
    seed=42,
    policy=policy,
)
initial_circuit = build_circuit({name: value.detach() for name, value in model.parameter_groups.items()})
fig, ax = initial_circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
for name, parameter in model.named_parameters():
    print(name, parameter.detach())
print("Initial prediction:", model().detach())


## 3. 选择模型要学习的目标

目标值为 0.25。损失是预测值与目标值之差的平方。
**运行前想一想：**损失为零意味着什么？

**概念：** 训练的是 RY 门的角度，而不是量子比特数量。计算关系是 `theta → cos(theta) → (cos(theta) - 0.25)²`。

**预期输出：** 预测值是 [-1, 1] 内的 Z 期望值，损失应逐渐接近零。对这里的单比特测量，P(0)=(1+prediction)/2；预测值本身不是分类概率。

**常见误解：** 这里使用本地可微 CPU 模拟器。`backward()` 在本地计算梯度，并不会在每一步向真机提交任务。


In [ ]:
target = 0.25
prediction = model()
loss = (prediction - target).square().mean()
initial_loss = loss.item()
print("Prediction:", prediction.item())
print("Target:", target)
print("Loss:", initial_loss)

print("theta -> cos(theta) -> loss:", model.parameter_groups["theta"].detach().item(), prediction.item(), initial_loss)


## 4. 查看梯度

`backward()` 计算损失随角度变化的梯度。
优化器会利用梯度调整参数。每次计算前都应清除旧梯度，因为 PyTorch 默认会累积梯度。

`zero_grad()` 在新一轮反向传播前清除累积梯度，否则前几步的梯度也会参与本次更新。`detach()` 将用于显示或记录的张量与梯度图分离，不会关闭整个模型的训练能力。计算训练损失前不要对 `prediction` 调用 `detach()`。


In [ ]:
learning_rate = 0.08
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
optimizer.zero_grad()
loss.backward()
for name, parameter in model.named_parameters():
    assert parameter.grad is not None
    assert torch.isfinite(parameter.grad).all()
    print(name, "gradient:", parameter.grad)


## 5. 更新一次参数

这是第一次优化器更新。比较更新前后的预测值。
如果要重做整个实验，请从创建 `model` 的单元开始；对已经使用过的计算图再次调用 `backward()`，并不等于开始了一轮新的训练。


In [ ]:
losses = [initial_loss]
optimizer.step()
print("Prediction after one update:", model().detach().item())


## 6. 重复同样的过程

每次迭代都重新预测、计算损失、求梯度，然后更新角度。
前面已经更新过一次，再运行 79 次，总计就是 80 次。


In [ ]:
total_steps = 80
for step in range(1, total_steps):
    optimizer.zero_grad()
    prediction = model()
    loss = (prediction - target).square().mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().item())

final_prediction = model().detach().item()
final_loss = (final_prediction - target) ** 2
print("Initial loss:", initial_loss)
print("Final loss:", final_loss)
print("Final prediction:", final_prediction)
assert final_loss < 0.01


## 7. 观察学习过程

损失不一定每一步都下降。使用默认设置时，最终损失应小于 0.01。

**读懂输出：** 默认设置下，`Final prediction` 应接近 0.25，`Final loss` 应小于 0.01。这个例子中 Z=0.25 表示 P(0)=0.625、P(1)=0.375，而不是 25% 的分类概率。打印的梯度表示敏感程度，不是下一步角度。

**安全重复运行：** 重建 `model` 和 `optimizer` 才会重启训练；只再次运行循环会继续更新已有角度。


In [ ]:

plt.plot(losses)
plt.xlabel("Optimizer step")
plt.ylabel("Squared error")
plt.show()


## 尝试修改

将 `target` 改为 -1 到 1 之间的其他值，或者修改 `learning_rate`，然后重启内核并依次运行。
参数直接在本 notebook 中修改，无需编辑另一个 Python 文件。
为什么这个模型无法达到目标值 2？

更完整的示例可参考 `examples/quick_start.py`，其中一个优化器同时训练经典层与量子层。
云端示例是独立的远程实验；本训练循环不会自动把学习到的参数提交到 QPU。


## 带回家使用

训练方式与普通 PyTorch 一样：用可微参数构造电路，计算期望值，
反向传播，再更新参数。这个独立示例直接展示了梯度路径。

```python
import torch
import flagquantum as fq

theta = torch.nn.Parameter(torch.tensor(0.7))
optimizer = torch.optim.Adam([theta], lr=0.08)
for step in range(80):
    optimizer.zero_grad()
    circuit = fq.Circuit(1).ry(0, theta)
    prediction = fq.run(circuit, outputs=fq.expectation(fq.Z(0))).expectation()
    loss = (prediction - 0.25).square().mean()
    loss.backward()
    optimizer.step()
print("Prediction:", prediction.detach().item())
```

如果需要带有命名参数、可复用的 PyTorch 层，可像本课一样使用 `fq.Module`。
本地训练电路不等于在量子硬件上执行训练循环。
